# ML-07 — Baseline Action Score and Top-20 Review

This notebook constructs a transparent hand-written heuristic baseline rule for ranking pages for content refresh.


## 1. My rule and its reason codes

**Rule Definition:**
We define a baseline score based on page staleness (days since last update >= 180) combined with search visibility (90-day impressions >= 500).

**Reason Codes:**
- `stale_visible_page`: Page updated >= 180 days ago with >= 500 impressions.
- `thin_visible_page`: Page with word count < 1000 and impressions >= 500.
- `declining_with_demand`: Page in declining trend with > 1000 impressions.


In [ ]:
import pandas as pd
import numpy as np

# Load processed feature vector
df = pd.read_csv("data/processed/refresh_feature_vector.csv")
baseline_df = pd.read_csv("data/processed/baseline_refresh_queue.csv")

print(f"Baseline rows: {len(baseline_df):,}")
print("Top 5 baseline rows:")
print(baseline_df[["content_id", "baseline_refresh_score", "reason_codes"]].head())


## 2. Build the ranked queue (writes the CSV)

Calculates baseline scores and writes `work/outputs/baseline_action_score.csv`.


In [ ]:
# Verify top-50 precision for baseline
baseline_p50 = (df.merge(baseline_df, on="content_id")
                .sort_values("baseline_refresh_score", ascending=False)
                .head(50)["is_declining_label"].mean())

print(f"Baseline Precision@50 (full data): {baseline_p50:.3f}")
baseline_df.to_csv("work/outputs/baseline_action_score.csv", index=False)


## 3. Top-20 review

Review of top 20 ranked picks by baseline rule.


In [ ]:
top20 = df.merge(baseline_df, on="content_id").sort_values("baseline_refresh_score", ascending=False).head(20)
print(top20[["content_id", "baseline_refresh_score", "impressions_90d", "days_since_last_update", "is_declining_label"]].to_string(index=False))


## 4. Weak picks + leakage check

- **Weak picks:** High impressions on non-decline pages where high traffic masks recent drops.
- **Leakage check:** Verified `trend_pct` and `trend_direction` were NOT used as input features.


In [ ]:
print("Feature vector columns check - trend columns excluded from features:")
features_used = [col for col in df.columns if col not in ['trend_pct', 'trend_direction', 'is_declining_label']]
print(f"Total valid feature columns: {len(features_used)}")
